In [ ]:
from pathlib import Path
import os
import pandas as pd
from src.agent_profiles import Agent, base_agent_options, proposer_options, skill_generator_options
from src.schemas import AgentResponse, ProposerResponse, ToolGeneratorResponse
from src.registry import ProgramManager, ProgramConfig
from src.agent_profiles.skill_generator import get_project_root
dataset_path = Path("~/mnt/shared-resources-sentient-research/salah_resources/datasets/officeqa/")
csv_path = os.path.join(dataset_path, "officeqa.csv")

train_data = pd.read_csv(csv_path)
hard_data = train_data[train_data['difficulty'] == 'hard']
easy_data = train_data[train_data['difficulty'] == 'easy']

In [2]:
sample_question = hard_data[hard_data.uid == "UID0123"]
question = sample_question.question.values[0]
answer = sample_question.answer.values[0]

### Initialize agents and manager

In [ ]:
# Create agents
base_agent = Agent(base_agent_options, AgentResponse)
proposer = Agent(proposer_options, ProposerResponse)
skill_generator = Agent(skill_generator_options, ToolGeneratorResponse)

# Initialize program manager
manager = ProgramManager(cwd=get_project_root())

### 1. Create base program branch

In [ ]:
base_config = ProgramConfig(
    name="base",
    parent=None,
    generation=0,
    system_prompt=base_agent_options.system_prompt,
    allowed_tools=base_agent_options.allowed_tools,
    output_format=base_agent_options.output_format,
    metadata={}
).with_timestamp()

manager.create_program("base", base_config)
print(f"Created program/base, branch: {manager._git_current_branch()}")

### 2. Run base agent

In [ ]:
# Switch to base program
manager.switch_to("base")

# Run agent
agent_trace = await base_agent.run(question)
print(f"Agent answer: {agent_trace.output.final_answer}")
print(f"Ground truth: {answer}")
print(f"Cost: ${agent_trace.total_cost_usd:.4f}")

### 3. Run proposer

In [ ]:
# Read feedback history
feedback_path = os.path.join(get_project_root(),".claude/feedback_history.md")
feedback_history = feedback_path.read_text() if feedback_path.exists() else "No previous attempts."

proposer_query = f"""## Previous Attempts
{feedback_history}

## Current Attempt
Agent trace: {agent_trace.output}
Agent Answer: {agent_trace.output.final_answer}
Ground Truth: {answer}"""

proposer_trace = await proposer.run(proposer_query)
print(f"Proposed: {proposer_trace.output.proposed_tool_or_skill}")
print(f"Justification: {proposer_trace.output.justification}")

# Append to feedback history
iteration_name = "iter-1"
entry = f"""
## {iteration_name}
**Skill**: {proposer_trace.output.proposed_tool_or_skill}
**Justification**: {proposer_trace.output.justification}\n\n
"""
with open(feedback_path, "a") as f:
    f.write(entry)

### 4. Create child program branch (iter-1)

In [ ]:
iter1_config = base_config.mutate("iter-1")
manager.create_program("iter-1", iter1_config, parent="base")
print(f"Created program/iter-1, branch: {manager._git_current_branch()}")

### 5. Run skill generator

In [ ]:
skill_query = f"""Proposed tool or skill: {proposer_trace.output.proposed_tool_or_skill}

Justification: {proposer_trace.output.justification}"""

skill_trace = await skill_generator.run(skill_query)
print(f"Generated skill: {skill_trace.output.generated_skill}")
print(f"Reasoning: {skill_trace.output.reasoning}")

### 6. Commit skills

In [ ]:
committed = manager.commit(f"iter-1: {proposer_trace.output.proposed_tool_or_skill[:50]}")
print(f"Committed: {committed}")
print(f"Lineage: {manager.get_lineage('iter-1')}")

### 7. Cleanup

In [ ]:
# Switch back to main
manager._git_checkout("main")
print(f"Back on: {manager._git_current_branch()}")

# Optionally discard test branches
# manager.discard("iter-1")
# manager.discard("base")